# Demo Arrow Flight: Executed Walkthrough

This notebook captures real outputs from high-level demo runs and grouped benchmarks.

## Run Context

- Captured on: 2026-04-07T09:55:29.825763
- Environment: repository-root execution
- Note: local machine-specific paths are intentionally omitted.

## High-Level Demos

### Demo

```bash
uv run poe demo
```

In [ ]:
uv run poe demo


Poe => uv run demo-arrow-flight roundtrip-one
Roundtrip one successful: location=grpc://127.0.0.1:52733, key=demo-image, shape(T,C,Z,Y,X)=(1, 1, 1, 96, 128)


### Demo Column

```bash
uv run poe demo_column
```

In [ ]:
uv run poe demo_column


Poe => uv run demo-arrow-flight roundtrip-column --rows 8 --height 32 --width 32 --seed 19
Roundtrip column successful: location=grpc://127.0.0.1:52739, key=demo-column, rows=8, columns=2, first_shape(T,C,Z,Y,X)=(1, 1, 1, 32, 32)


### Parquet Demo

```bash
uv run demo-arrow-flight parquet-demo --output docs/notebooks/artifacts/random_ome_dataset.parquet --rows 10 --height 64 --width 64 --seed 11 --batch-rows 3
```

In [ ]:
uv run demo-arrow-flight parquet-demo --output docs/notebooks/artifacts/random_ome_dataset.parquet --rows 10 --height 64 --width 64 --seed 11 --batch-rows 3


Parquet stream demo successful: dataset=docs/notebooks/artifacts/random_ome_dataset.parquet, sent_rows=10, sent_chunks=4, received_chunks=4, chunk_rows=[3, 3, 3, 1], received_rows=10


### Pipeline Demo

```bash
uv run poe pipeline_demo
```

In [ ]:
uv run poe pipeline_demo


Poe => uv run demo-arrow-flight pipeline-demo --rows 10 --height 32 --width 32 --seed 17 --stage-name preprocess
Pipeline demo successful: location=grpc://127.0.0.1:52753, raw_key=pipeline-raw, processed_key=pipeline-processed, produced_rows=10, transformed_rows=10, consumed_rows=10, chunk_rows=[10]


### Slurm Simulate

```bash
uv run demo-arrow-flight slurm-simulate --output-dir docs/notebooks/artifacts/slurm_sim --rows 10 --height 32 --width 32 --seed 101 --batch-rows 3
```

In [ ]:
uv run demo-arrow-flight slurm-simulate --output-dir docs/notebooks/artifacts/slurm_sim --rows 10 --height 32 --width 32 --seed 101 --batch-rows 3


Slurm simulation successful: location=grpc://127.0.0.1:8825, dataset=docs/notebooks/artifacts/slurm_sim/random_ome_dataset.parquet, job_ids=[866061, 289852, 186843], sent_rows=10, chunk_rows=[3, 3, 3, 1]


## Grouped Benchmarking

### Single Benchmark Run

```bash
uv run demo-arrow-flight benchmark-demo --output docs/notebooks/artifacts/random_ome_benchmark.parquet --rows 20 --height 64 --width 64 --seed 29 --batch-rows 4 --repeats 3 --output-csv docs/notebooks/artifacts/benchmark_demo.csv
```

In [ ]:
uv run demo-arrow-flight benchmark-demo --output docs/notebooks/artifacts/random_ome_benchmark.parquet --rows 20 --height 64 --width 64 --seed 29 --batch-rows 4 --repeats 3 --output-csv docs/notebooks/artifacts/benchmark_demo.csv


Benchmark complete: baseline_seconds=0.0136, flight_seconds=0.0409, baseline_mib_s=150.31, flight_mib_s=50.11, flight_vs_baseline_time_ratio=2.999, output_csv=docs/notebooks/artifacts/benchmark_demo.csv
Benchmark overhead complete: parquet_write_read_seconds=0.2463, flight_roundtrip_seconds=0.0167, parquet_mib_s=3.87, flight_mib_s=57.13, parquet_vs_flight_time_ratio=14.764, output_csv=docs/notebooks/artifacts/benchmark_demo_overhead.csv


### Scaling Study (6 Larger Data Sizes)

Rows tested: `160, 320, 640, 1280, 2560, 5120` (fixed image shape `64x64`).

| rows | parquet_size_kib | baseline_seconds | flight_seconds | baseline_mib_s | flight_mib_s |
| --- | --- | --- | --- | --- | --- |
| 160.0 | 3092.83 | 0.010011 | 0.01996 | 301.703 | 151.317 |
| 320.0 | 5659.43 | 0.015318 | 0.031841 | 360.811 | 173.574 |
| 640.0 | 10792.71 | 0.033174 | 0.141793 | 317.709 | 74.332 |
| 1280.0 | 21059.42 | 0.073512 | 0.180752 | 279.761 | 113.78 |
| 2560.0 | 41594.12 | 0.152038 | 0.22169 | 267.164 | 183.225 |
| 5120.0 | 82666.15 | 0.259623 | 0.573434 | 310.946 | 140.781 |

### Scaling Plot

![Benchmark Scaling](artifacts/benchmark_scaling.png)

## Many-Batch Pipeline Proof (I/O Avoidance)\n\nThis section compares many-record to many-record processing in two modes as batch count doubles:\n\n- `file pipeline`: parquet intermediate write/read between stages\n- `flight pipeline`: in-memory Flight key handoff between stages (no intermediate file writes)

In [ ]:
uv run demo-arrow-flight benchmark-pipeline-io --batch-counts 160,320,640,1280,2560,5120 --batch-rows 1 --height 64 --width 64 --seed 41 --repeats 1 --output-csv docs/notebooks/artifacts/pipeline_io_scaling.csv


Pipeline I/O benchmark complete: location=grpc://127.0.0.1:<port>, output_csv=docs/notebooks/artifacts/pipeline_io_scaling.csv, batch_counts=[160, 320, 640, 1280, 2560, 5120]\n

### Pipeline Scaling Table and Plot

This benchmark compares:

- `file pipeline`: parquet intermediate write/read between stages
- `flight pipeline`: in-memory Flight key handoff between stages

### Pipeline Scaling Plot

![Pipeline I/O Scaling](artifacts/pipeline_io_scaling.png)


## Repeated Per-Batch I/O Stress Test

This is the explicit repeated handoff benchmark:

- file mode: every batch writes/reads parquet intermediates
- flight mode: every batch uses Flight key handoff

This is separate from the existing scaling plot and is designed to expose repeated I/O overhead.

### Repeated I/O Scaling Table

This table/plot is retained as the original repeated per-batch stress test artifact.


### Repeated I/O Scaling Plot

![Repeated Pipeline I/O Scaling](artifacts/pipeline_io_repeated_scaling.png)

## Analysis\n\n1. The demos succeed end-to-end with Arrow-native transfer paths.\n2. Chunked transport remains visible in parquet and slurm simulation flows.\n3. The benchmark section now includes a direct overhead comparison: parquet write+read vs Flight roundtrip on the same Arrow table.\n4. On this single node, results can vary by mode and data size; local scans may still be faster than networked transport for some cases.\n5. For cluster planning, run the same benchmark workflow on separated server/client nodes and compare trendline slopes plus overhead ratio.